# AI Programming Foundations — Data Workflow

**Student:** Maurizio Pinto  
**Dataset:** Titanic - Machine Learning from Disaster  
**Source:** https://www.kaggle.com/c/titanic/data  

This project builds a complete, reproducible data workflow using Python. I load the Titanic dataset, clean and transform it, perform exploratory analysis, create visualizations, and communicate findings. This workflow serves as the foundation for future ML, deep learning, and agentic AI projects in the capstone.

## Setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")
%matplotlib inline

## Data Ingestion

In [ ]:
df = pd.read_csv("dataset/train.csv")
print(f"Dataset shape: {df.shape}")
df.head()

In [ ]:
df.info()

In [ ]:
print("Missing values per column:")
print(df.isnull().sum())

## Data Cleaning

In [ ]:
def fill_missing_age(df: pd.DataFrame) -> pd.DataFrame:
    """Fill missing Age values using median age grouped by Pclass and Sex.

    Grouping by passenger class and sex preserves the correlation between
    these variables and age, producing more accurate imputations than a
    simple global median.

    Parameters
    ----------
    df : pd.DataFrame
        Titanic dataset with columns 'Age', 'Pclass', and 'Sex'.

    Returns
    -------
    pd.DataFrame
        DataFrame with all missing Age values filled.
    """
    df = df.copy()
    df["Age"] = df.groupby(["Pclass", "Sex"])["Age"].transform(
        lambda x: x.fillna(x.median())
    )
    return df


def clean_cabin_and_embarked(df: pd.DataFrame) -> pd.DataFrame:
    """Drop sparse columns, fill Embarked, and remove non-analytical columns.

    Drops the Cabin column due to excessive missingness (~77%), fills the
    two missing Embarked values with the mode, and removes PassengerId,
    Name, and Ticket as they do not contribute to exploratory analysis.

    Parameters
    ----------
    df : pd.DataFrame
        Titanic dataset after age imputation.

    Returns
    -------
    pd.DataFrame
        Cleaned DataFrame with only analytically useful columns.
    """
    df = df.copy()
    df = df.drop(columns=["Cabin"])
    df["Embarked"] = df["Embarked"].fillna(df["Embarked"].mode()[0])
    df = df.drop(columns=["PassengerId", "Name", "Ticket"])
    return df

In [ ]:
print(f"Before cleaning: {df.shape}")
print(df.isnull().sum())
print()

df = fill_missing_age(df)
df = clean_cabin_and_embarked(df)

print(f"After cleaning: {df.shape}")
print(df.isnull().sum())
df.head()

## Exploratory Data Analysis

In [ ]:
def explore_data(df: pd.DataFrame) -> dict:
    """Run exploratory analysis on the cleaned Titanic dataset.

    Produces summary statistics, survival rate breakdowns by key
    demographic groups, a correlation matrix for numeric columns,
    and age distribution statistics stratified by survival status.

    Parameters
    ----------
    df : pd.DataFrame
        Cleaned Titanic dataset.

    Returns
    -------
    dict
        Dictionary containing DataFrames for each analysis result.
    """
    results = {}

    results["summary_stats"] = df.describe()
    results["survival_by_class"] = df.groupby("Pclass")["Survived"].mean()
    results["survival_by_sex"] = df.groupby("Sex")["Survived"].mean()
    results["survival_by_embarked"] = df.groupby("Embarked")["Survived"].mean()
    results["correlation"] = df.select_dtypes(include=[np.number]).corr()
    results["age_by_survival"] = df.groupby("Survived")["Age"].describe()

    return results

In [ ]:
results = explore_data(df)

print("=== Summary Statistics ===")
display(results["summary_stats"])

print("\n=== Survival Rate by Passenger Class ===")
print(results["survival_by_class"])

print("\n=== Survival Rate by Sex ===")
print(results["survival_by_sex"])

print("\n=== Survival Rate by Embarkation Port ===")
print(results["survival_by_embarked"])

print("\n=== Age Statistics by Survival ===")
display(results["age_by_survival"])

print("\n=== Correlation Matrix ===")
display(results["correlation"])

## Visualizations

In [ ]:
# Figure 1: Survival Rate by Passenger Class and Sex
survival_rates = df.groupby(["Pclass", "Sex"])["Survived"].mean().reset_index()

fig, ax = plt.subplots(figsize=(8, 5))
sns.barplot(
    data=survival_rates,
    x="Pclass",
    y="Survived",
    hue="Sex",
    palette="Set2",
    ax=ax,
)
ax.set_title("Figure 1: Survival Rate by Passenger Class and Sex")
ax.set_xlabel("Passenger Class")
ax.set_ylabel("Survival Rate")
ax.set_ylim(0, 1)
plt.tight_layout()
plt.savefig("figure1_survival_by_class_sex.png", dpi=150)
plt.show()

**Figure 1 Interpretation:** Female passengers had significantly higher survival rates than male passengers across all classes. First-class females had the highest survival rate (~97%), while third-class males had the lowest (~14%). Class and gender together strongly influenced survival outcomes.

In [ ]:
# Figure 2: Age Distribution by Survival Status
fig, ax = plt.subplots(figsize=(8, 5))
sns.kdeplot(
    data=df,
    x="Age",
    hue="Survived",
    fill=True,
    common_norm=False,
    palette="Set2",
    ax=ax,
)
ax.set_title("Figure 2: Age Distribution by Survival Status")
ax.set_xlabel("Age (years)")
ax.set_ylabel("Density")
ax.legend(title="Survived", labels=["No", "Yes"])
plt.tight_layout()
plt.savefig("figure2_age_by_survival.png", dpi=150)
plt.show()

**Figure 2 Interpretation:** Children (ages 0–10) show a distinct survival advantage compared to other age groups. The distributions for survivors and non-survivors are otherwise similar, peaking around age 20–30. This suggests that the "women and children first" protocol was applied during evacuation.

In [ ]:
# Figure 3: Fare Distribution by Passenger Class
fig, ax = plt.subplots(figsize=(8, 5))
sns.boxplot(
    data=df,
    x="Pclass",
    y="Fare",
    hue="Pclass",
    palette="Set2",
    ax=ax,
    legend=False,
)
ax.set_title("Figure 3: Fare Distribution by Passenger Class")
ax.set_xlabel("Passenger Class")
ax.set_ylabel("Fare (£)")
plt.tight_layout()
plt.savefig("figure3_fare_by_class.png", dpi=150)
plt.show()

**Figure 3 Interpretation:** First-class fares were substantially higher and more variable than second and third class. First class has several outliers above £200, with one fare exceeding £500. The economic stratification visible here aligns with the survival advantage observed in Figure 1, as wealthier passengers had better cabin locations and earlier access to lifeboats.

## Summary and Interpretation

This analysis of the Titanic dataset revealed several key patterns:

**Key Insights**
- **Gender was the strongest survival predictor.** Female passengers survived at much higher rates than males across all classes (e.g., ~97% vs ~37% in first class).
- **Class mattered significantly.** First-class passengers had a survival rate of approximately 63%, compared to ~47% for second class and ~24% for third class.
- **Children were prioritized.** The age distribution shows a clear survival advantage for passengers under 10, consistent with evacuation protocols.
- **Fare reflects economic stratification.** First-class fares were dramatically higher with large outliers, correlating with both cabin location and survival.

**Limitations**
- The test set does not include the `Survived` column, so this analysis is limited to the 891 training passengers only.
- The `Cabin` column was dropped due to 77% missingness, removing potentially useful deck-location information.
- Age imputation by group median may mask individual variation and could introduce bias if the missingness pattern is not random.
- This dataset captures only a subset of passengers and crew; the full passenger manifest is larger.

**Surprising Observations**
- Embarkation port shows a survival difference (Cherbourg passengers survived at higher rates), which likely reflects class composition rather than a direct effect of the port itself.
- The wide spread of first-class fares suggests economic diversity even within the highest class.